## Usefull libraries:

In [198]:
import numpy as np
import glob
import camb
from camb import model, initialpower
import math
import matplotlib.pyplot as plt
from scipy import special
from astropy.io import fits
from astropy import units as u
from astropy.coordinates import SkyCoord
from scipy.integrate import quad
from scipy.interpolate import interp1d
from scipy.integrate import simpson
from scipy.integrate import trapezoid
from CosmoFunc import *
from BF_OptMC import *
from lu_functions import *
import time
import os
from numba import jit, prange

## Extract the R_matrix from the .txt file

In [ ]:
# paramètres:
N = 2500 # nombre de galaxies qu'on prend


# Pattern pour capturer tous les fichiers avec ce format
fichiers = sorted(glob.glob(f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_{N}g_sample_*.txt'))

# Extraire toutes les matrices
R = []
for fichier in fichiers:
    mat = np.loadtxt(fichier, skiprows=2)
    R.append(mat)
    print(f"✓ Chargé: {fichier} -> forme {mat.shape}")

# Stocker dans des variables R1, R2, etc.
for i, mat in enumerate(R, 1):
    globals()[f'R{i}'] = mat


✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_1.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_10.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_3.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_4.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_6.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_7.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_8.txt -> forme (9, 9)
✓ Chargé: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_2500g_sample_9.txt -> forme (9, 9)


##  Statistics on R

In [200]:
# Calculer la matrice moyenne (covariance moyenne)
R_mean = np.mean(R, axis=0)

# Vous pouvez aussi calculer l'écart-type entre les runs
R_std = np.std(R, axis=0)

# Erreur standard de la moyenne (incertitude sur la moyenne)
R_sem = R_std / np.sqrt(len(R))  # 10 runs


# R_thorie est la resulats de Fei:

R_theorie = np.array([
    [5248.882, -58.604, -2725.658, -1.093, 1.034, -9.032, -0.357, 0.153, 12.569],
    [-58.604, 7048.148, 1998.970, 3.336, 0.974, 0.819, 1.109, -13.604, -9.018],
    [-2725.658, 1998.970, 45434.716, 30.027, 3.378, 18.922, 17.950, -0.134, -203.080],
    [-1.093, 3.336, 30.027, 0.140, 0.005, 0.012, 0.061, -0.005, -0.093],
    [1.034, 0.974, 3.378, 0.005, 0.036, -0.002, 0.005, 0.003, -0.016],
    [-9.032, 0.819, 18.922, 0.012, -0.002, 0.085, 0.014, 0.003, -0.085],
    [-0.357, 1.109, 17.950, 0.061, 0.005, 0.014, 0.152, 0.003, -0.042],
    [0.153, -13.604, -0.134, -0.005, 0.003, 0.003, 0.003, 0.098, -0.003],
    [12.569, -9.018, -203.080, -0.093, -0.016, -0.085, -0.042, 0.003, 1.103]])

# Calculer la différence normalisée par l'erreur
difference_normalisee = (R_theorie - R_mean) / R_sem


## Enregistrer la matrice moyenne/ ecart-type et celle de Fei

In [201]:
with open(f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_random_moy_{N}.txt', 'w') as f:
    f.write(f"R matrix mean with 10 samples with {N} galaxies each :\n\n")
    for row in R_mean:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")
    f.write(f"R matrix ecart-type : \n\n")
    for row in R_std:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")  
    f.write(f"R matrix standard error on the mean :\n\n")        
    for row in R_sem:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")  
    f.write("(R_Fei - R_mean) / R_error :\n\n")
    for row in difference_normalisee:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")       

print(f"Results saved in: {f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_random_moy_{N}.txt'}")


with open(f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_exact_Fei.txt', 'w') as f:
    f.write(f"R matrix exact from Fei Qin :\n\n")
    for row in R_theorie:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")
        
print(f"Results saved in:{f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_exact_Fei.txt'}")


Results saved in: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_random_moy_2500.txt
Results saved in:/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_exact_Fei.txt


## Extraire la prédiction théorique sur le Bulk flow

In [202]:
# We can extract the Bulk flow from the R matrix with the jacobian matrix:

V_bkf = [0,0,0] # setting the Bulk flow 
B_pq = R_mean[:3, :3] # we take only the first 3 rows and 3 columns of the R matrix since we want to extract only the Bulk flow and not the shear

np.set_printoptions(precision=6, suppress=True)
print('R_pq part of the matrix with the Bulk Flow component:'),
for row in B_pq:
    print("  ".join(f"{x:10.4f}" for x in row))

R_pq part of the matrix with the Bulk Flow component:
 5344.8397      1.4784  -2604.8143
    1.4784   7634.5110   2827.4235
-2604.8143   2827.4235  48052.2749


In [203]:
# extract the theoritical Bulk Flow from the sigma_B: 
# sigma_B est la racine de la trace de la matrice 3*3 extraite de R_pq:
# la prediction de lambda CDM est : B = 0 +- (2/3)**1/2 * sigma_B
trace_B_pq = np.trace(B_pq)
sigma_B = np.sqrt(trace_B_pq)
B_p= np.sqrt(2/3)*sigma_B
print('The theoritical Bulk Flow from LambdaCDM is : 0 +-', sigma_B, 'km/s')
print('Bp = (2/3)**1/2 * sigmaB = ',B_p,'km/s')

The theoritical Bulk Flow from LambdaCDM is : 0 +- 247.0457964376241 km/s
Bp = (2/3)**1/2 * sigmaB =  201.71204812388706 km/s


## extraire l'amplitude du bulk flow de nos estimateurs

In [ ]:
# B = np.sqrt( Bx**2 + By**2 + Bz**2)
# pour l'estimateur etaMLE:
# We import the bulk flow results from the .txt files
Input_dir   = '/renoir/fromenti/Documents/codes_Bulk_flow/results/'
Fit_tech1 = 'etaMLE'  # Fit_tech can be 'etaMLE', 'wMLE' or 'tMLE'


# We load the bulk flow results of the data:
Bx_data_eta, By_data_eta, Bz_data_eta = np.loadtxt(Input_dir + f'Bulk_flow_data_{Fit_tech1}.txt',skiprows=6,usecols=1,max_rows=3)
# we load the error on the Bx,By,Bz estimation for the data:
err_Bx_eta, err_By_eta ,err_Bz_eta = np.loadtxt( Input_dir + f'Bulk_flow_data_{Fit_tech1}.txt', skiprows=6, usecols=2, max_rows=3)

B_data_eta = np.sqrt( Bx_data_eta**2 + By_data_eta**2 + Bz_data_eta**2)
# et on fait la propagation des incertitudes sur B:
err_B_eta = (np.abs(Bx_data_eta*err_Bx_eta) + np.abs(By_data_eta*err_By_eta) + np.abs(Bz_data_eta*err_Bz_eta)) / B_data_eta

# load the file with the mocks values of BF
all_mocks_eta = np.loadtxt(Input_dir + f'Bulk_flow_all_mocks_{Fit_tech1}.txt', skiprows=1)
# we extract the values of Bx,By,Bz and the true values of Ux, Uy, Uz for all the mocks:
Bx_mocks_eta = all_mocks_eta[:, 1]  
By_mocks_eta = all_mocks_eta[:, 2] 
Bz_mocks_eta = all_mocks_eta[:, 3]  

# 1. Calculez l'amplitude B pour chaque mock
B_mocks_eta = np.sqrt(Bx_mocks_eta**2 + By_mocks_eta**2 + Bz_mocks_eta**2)
# 2. L'erreur est l'écart-type de cette distribution
err_B_mocks_eta = np.std(B_mocks_eta, ddof=1)

print('For estimator etaMLE the Bulk Flow with propagation formula is:',B_data_eta,'+-',err_B_eta,'km/s')
print('For estimator etaMLE the Bulk Flow with mocks is:',B_data_eta,'+-',err_B_mocks_eta,'km/s')


For estimator etaMLE the Bulk Flow with propagation formula is: 377.95160964536075 +- 43.6572591114911 km/s


In [ ]:
# B = np.sqrt( Bx**2 + By**2 + Bz**2)
# pour l'estimateur wMLE:
# We import the bulk flow results from the .txt files
Input_dir   = '/renoir/fromenti/Documents/codes_Bulk_flow/results/'
Fit_tech2 = 'wMLE'  # Fit_tech can be 'etaMLE', 'wMLE' or 'tMLE'

# We load the bulk flow results of the data:
Bx_data_w, By_data_w, Bz_data_w = np.loadtxt(Input_dir + f'Bulk_flow_data_{Fit_tech2}.txt',skiprows=6,usecols=1,max_rows=3)
# we load the error on the Bx,By,Bz estimation for the data:
err_Bx_w, err_By_w ,err_Bz_w = np.loadtxt( Input_dir + f'Bulk_flow_data_{Fit_tech2}.txt', skiprows=6, usecols=2, max_rows=3)

B_data_w = np.sqrt( Bx_data_w**2 + By_data_w**2 + Bz_data_w**2)
# et on fait la propagation des incertitudes sur B:
err_B_w = (np.abs(Bx_data_w*err_Bx_w) + np.abs(By_data_w*err_By_w) + np.abs(Bz_data_w*err_Bz_w)) / B_data_w

# load the file with the mocks values of BF
all_mocks_w = np.loadtxt(Input_dir + f'Bulk_flow_all_mocks_{Fit_tech2}.txt', skiprows=1)

# we extract the values of Bx,By,Bz and the true values of Ux, Uy, Uz for all the mocks:
Bx_mocks_w = all_mocks_w[:, 1]  
By_mocks_w = all_mocks_w[:, 2] 
Bz_mocks_w = all_mocks_w[:, 3]  

# 1. Calculez l'amplitude B pour chaque mock
B_mocks_w = np.sqrt(Bx_mocks_w**2 + By_mocks_w**2 + Bz_mocks_w**2)
# 2. L'erreur est l'écart-type de cette distribution
err_B_mocks_w = np.std(B_mocks_w, ddof=1)

print('For estimator etaMLE the Bulk Flow with propagation formula is:',B_data_w,'+-',err_B_w,'km/s')
print('For estimator etaMLE the Bulk Flow with mocks is:',B_data_w,'+-',err_B_mocks_w,'km/s')

For estimator wMLE the Bulk Flow with propagation formula is: 386.6625539008687 +- 44.684122235909754 km/s


In [ ]:
# B = np.sqrt( Bx**2 + By**2 + Bz**2)
# pour l'estimateur tMLE:
# We import the bulk flow results from the .txt files
Input_dir   = '/renoir/fromenti/Documents/codes_Bulk_flow/results/'
Fit_tech3 = 'tMLE'  # Fit_tech can be 'etaMLE', 'wMLE' or 'tMLE'

# We load the bulk flow results of the data:
Bx_data_t, By_data_t, Bz_data_t = np.loadtxt(Input_dir + f'Bulk_flow_data_{Fit_tech3}.txt',skiprows=6,usecols=1,max_rows=3)
# we load the error on the Bx,By,Bz estimation for the data:
err_Bx_t, err_By_t ,err_Bz_t = np.loadtxt( Input_dir + f'Bulk_flow_data_{Fit_tech3}.txt', skiprows=6, usecols=2, max_rows=3)

B_data_t = np.sqrt( Bx_data_t**2 + By_data_t**2 + Bz_data_t**2)
# et on fait la propagation des incertitudes sur B:
err_B_t  = (np.abs(Bx_data_t*err_Bx_t) + np.abs(By_data_t*err_By_t) + np.abs(Bz_data_t*err_Bz_t)) / B_data_t

# load the file with the mocks values of BF
all_mocks_t = np.loadtxt(Input_dir + f'Bulk_flow_all_mocks_{Fit_tech3}.txt', skiprows=1)

# we extract the values of Bx,By,Bz and the true values of Ux, Uy, Uz for all the mocks:
Bx_mocks_t = all_mocks_t[:, 1]  
By_mocks_t = all_mocks_t[:, 2] 
Bz_mocks_t = all_mocks_t[:, 3]  

# 1. Calculez l'amplitude B pour chaque mock
B_mocks_t = np.sqrt(Bx_mocks_t**2 + By_mocks_t**2 + Bz_mocks_t**2)
# 2. L'erreur est l'écart-type de cette distribution
err_B_mocks_t = np.std(B_mocks_t, ddof=1)

print('For estimator etaMLE the Bulk Flow with propagation formula is:',B_data_t,'+-',err_B_t,'km/s')
print('For estimator etaMLE the Bulk Flow with mocks is:',B_data_t,'+-',err_B_mocks_t,'km/s')

For estimator tMLE the Bulk Flow with propagation formula is: 964.4515275939416 +- 46.619667152339275 km/s


In [ ]:
# chi² entre la prédiction de LamdaCDM et les vrais data:   χ2 = Up ​(Cpq ​+ Rpqv​)**-1 Uq**T
# première etape extraire C_pq la matrice de covariance des B_datas:
#B_data, B_MC, Sv_MC, Um_Cov, SX, SY, SZ, SS = bulk_flow_data_calculator('/renoir/fromenti/Documents/data_DESI/combinedpv/Y1/PV_clustering_data_v5_v13.fits', Fit_tech = Fit_tech1)

In [208]:
def chi2(B_measured,B_pq, C_pq):
    
    B_measured_transposée = B_measured.T
    Tot_pq = B_pq + C_pq
    Tot_pq_inv = np.linalg.inv(Tot_pq)
    # ici la matrice B_measured est la matrice des datas
    chi2 = B_measured @ Tot_pq_inv @ B_measured_transposée
    
    return chi2

C_pq_etaMLE = np.array([[ 476.656962 , -85.661819 , 177.999417],  # je l'ai réecrite ici afin de ne pas refaire tourner le bulk_flow_calculator à chaque fois
 [ -85.661819 , 770.201498,  223.915886],
 [ 177.999417  ,223.915886 ,1340.070206]])

B_eta = np.array([Bx_data_eta,By_data_eta,Bz_data_eta]) # la matrice B qui est comme Up mais ici on prend que le bulk flow par simplicité
chi2_norm_eta = chi2(B_eta,B_pq,C_pq_etaMLE) / 3 
  # on normalise le chi2
print('The chi2 of the prediction of the bulk flow normalize versus the data for etaMLE is:',chi2_norm_eta)

The chi2 of the prediction of the bulk flow normalize versus the data for etaMLE is: 4.808686300885391


In [209]:
# chi² entre la prédiction de LamdaCDM et les vrais data:   χ2 = Up ​(Cpq ​+ Rpqv​)**-1 Uq**T
# première etape extraire C_pq la matrice de covariance des B_datas: avec le wMLE estimator
#B_data_w, B_MC, Sv_MC, Um_Cov, SX, SY, SZ, SS = bulk_flow_data_calculator('/renoir/fromenti/Documents/data_DESI/combinedpv/Y1/PV_clustering_data_v5_v13.fits', Fit_tech = Fit_tech2)
#print(Um_Cov)

In [210]:
C_pq_wMLE = np.array([[ 503.852497 , -56.177918 , 206.728807],  # je l'ai réecrite ici afin de ne pas refaire tourner le bulk_flow_calculator à chaque fois
 [ -56.177918  ,723.424403  ,179.833793],
 [1206.728807 , 179.833793 ,1317.999919 ]])

B_w = np.array([Bx_data_w,By_data_w,Bz_data_w]) # la matrice B qui est comme Up mais ici on prend que le bulk flow par simplicité
chi2_norm_w = chi2(B_w,B_pq,C_pq_wMLE) / 3 

print('The chi2 of the prediction od the bulk flow normalize versus the data for wMLE is:',chi2_norm_w)

The chi2 of the prediction od the bulk flow normalize versus the data for wMLE is: 4.882981145276894


## Extraire la prédiction de LambdaCDM sur le shear moment

In [220]:
# We can extract the Shear moment from the R matrix with the jacobian matrix:

V_bkf = [0,0,0] # setting the Bulk flow 
R_shear = R_mean[3:9, 3:9] # we take only the first 3 rows and 3 columns of the R matrix since we want to extract only the Bulk flow and not the shear

np.set_printoptions(precision=6, suppress=True)
print('R_pq part of the matrix with the shear momoent component:'),
for row in R_shear:
    print("  ".join(f"{x:10.4f}" for x in row))

R_pq part of the matrix with the shear momoent component:
    0.1372      0.0056      0.0068      0.0597     -0.0030     -0.0874
    0.0056      0.0386     -0.0007      0.0038      0.0035     -0.0214
    0.0068     -0.0007      0.0945      0.0129      0.0041     -0.0756
    0.0597      0.0038      0.0129      0.1668      0.0048     -0.0522
   -0.0030      0.0035      0.0041      0.0048      0.1100      0.0106
   -0.0874     -0.0214     -0.0756     -0.0522      0.0106      1.2143


In [235]:
# or la compo Qzz est calculer ensuite pour les datas donc on va enlever la ligne lié à cet element qui est en dernière position si on suit l'ordre pris par la g 
R_shear = R_mean[3:8, 3:8] # we take only the first 3 rows and 3 columns of the R matrix since we want to extract only the Bulk flow and not the shear

np.set_printoptions(precision=6, suppress=True)
print('R_pq part of the matrix with the shear momoent component without Qzz:'),
for row in R_shear:
    print("  ".join(f"{x:10.4f}" for x in row))

R_pq part of the matrix with the shear momoent component without Qzz:
    0.1372      0.0056      0.0068      0.0597     -0.0030
    0.0056      0.0386     -0.0007      0.0038      0.0035
    0.0068     -0.0007      0.0945      0.0129      0.0041
    0.0597      0.0038      0.0129      0.1668      0.0048
   -0.0030      0.0035      0.0041      0.0048      0.1100


In [243]:
# la prediction de lambda CDM est : Q = 0 +- sigma_Q
# 2. Les CRMS sont les racines carrées des éléments diagonaux et on les exctrait dans le meme ordre que la matrice g du code lambdaCDM2_corrected.py
CRMS_Qxx = np.sqrt(R_shear[0, 0])  
CRMS_Qxy = np.sqrt(R_shear[1, 1])
CRMS_Qxz = np.sqrt(R_shear[2, 2])
CRMS_Qyy = np.sqrt(R_shear[3, 3])
CRMS_Qyz = np.sqrt(R_shear[4, 4])
CRMS_Qzz =np.sqrt( np.abs(- R_shear[0, 0] - R_shear[3, 3] ) )

print("Prédiction ΛCDM pour les shear moments :")
print(f"Qxx = 0 ± {CRMS_Qxx:.4f} km/s")
print(f"Qyy = 0 ± {CRMS_Qyy:.4f} km/s")
print(f"Qxy = 0 ± {CRMS_Qxy:.4f} km/s")
print(f"Qxz = 0 ± {CRMS_Qxz:.4f} km/s")
print(f"Qyy = 0 ± {CRMS_Qyy:.4f} km/s")
print(f"Qyz = 0 ± {CRMS_Qyz:.4f} km/s")
print(f"Qzz = 0 ± {CRMS_Qzz:.4f} km/s")

Prédiction ΛCDM pour les shear moments :
Qxx = 0 ± 0.3704 km/s
Qyy = 0 ± 0.4084 km/s
Qxy = 0 ± 0.1965 km/s
Qxz = 0 ± 0.3074 km/s
Qyy = 0 ± 0.4084 km/s
Qyz = 0 ± 0.3316 km/s
Qzz = 0 ± 0.5513 km/s


## Extraire les composantes du shear momoent des datas

In [227]:

# pour l'estimateur etaMLE:
# We import the shear moment results from the .txt files
Input_dir   = '/renoir/fromenti/Documents/codes_Bulk_flow/results/'
Fit_tech1 = 'etaMLE'  # Fit_tech can be 'etaMLE', 'wMLE' or 'tMLE'

# We load the bulk flow results of the data:
Qxx_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=8,usecols=1,max_rows=1)
Qxy_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=9,usecols=1,max_rows=1)
Qxz_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=10,usecols=1,max_rows=1)
Qyy_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=11,usecols=1,max_rows=1)
Qyz_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=12,usecols=1,max_rows=1)
Qzz_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=13,usecols=1,max_rows=1)

# we load the error on the Bx,By,Bz estimation for the data:
error_Qxx_eta = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=8, usecols=2, max_rows=1)
error_Qxy_eta = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=9, usecols=2, max_rows=1)
error_Qxz_eta = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=10, usecols=2, max_rows=1)
error_Qyy_eta = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=11, usecols=2, max_rows=1)
error_Qyz_eta = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=12, usecols=2, max_rows=1)
error_Qzz_eta = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=13, usecols=2, max_rows=1)


print('For estimator etaMLE the Qxx component of the shear moment is:',Qxx_data_eta,'+-',error_Qxx_eta,'km/s')
print('For estimator etaMLE the Qyy component of the shear moment is:',Qyy_data_eta,'+-',error_Qyy_eta,'km/s')
print('For estimator etaMLE the Qzz component of the shear moment is:',Qzz_data_eta,'+-',error_Qzz_eta,'km/s')
print('For estimator etaMLE the Qxy component of the shear moment is:',Qxy_data_eta,'+-',error_Qxy_eta,'km/s')
print('For estimator etaMLE the Qxz component of the shear moment is:',Qxz_data_eta,'+-',error_Qxz_eta,'km/s')
print('For estimator etaMLE the Qyz component of the shear moment is:',Qyz_data_eta,'+-',error_Qyz_eta,'km/s')

For estimator etaMLE the Qxx component of the shear moment is: 0.1302 +- 0.1782 km/s
For estimator etaMLE the Qyy component of the shear moment is: -0.8355 +- 0.2666 km/s
For estimator etaMLE the Qzz component of the shear moment is: 0.7054 +- 0.3206 km/s
For estimator etaMLE the Qxy component of the shear moment is: -0.0078 +- 0.1542 km/s
For estimator etaMLE the Qxz component of the shear moment is: -0.7917 +- 0.2879 km/s
For estimator etaMLE the Qyz component of the shear moment is: -0.4743 +- 0.2714 km/s


In [228]:

# pour l'estimateur wMLE:
# We import the shear moment results from the .txt files
Input_dir   = '/renoir/fromenti/Documents/codes_Bulk_flow/results/'
Fit_tech2 = 'wMLE'  # Fit_tech can be 'etaMLE', 'wMLE' or 'tMLE'

# We load the bulk flow results of the data:
Qxx_data_w = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=8,usecols=1,max_rows=1)
Qxy_data_w = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=9,usecols=1,max_rows=1)
Qxz_data_w = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=10,usecols=1,max_rows=1)
Qyy_data_w = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=11,usecols=1,max_rows=1)
Qyz_data_w = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=12,usecols=1,max_rows=1)
Qzz_data_w = np.loadtxt(Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt',skiprows=13,usecols=1,max_rows=1)

# we load the error on the Bx,By,Bz estimation for the data:
error_Qxx_w = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=8, usecols=2, max_rows=1)
error_Qxy_w = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=9, usecols=2, max_rows=1)
error_Qxz_w = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=10, usecols=2, max_rows=1)
error_Qyy_w = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=11, usecols=2, max_rows=1)
error_Qyz_w = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=12, usecols=2, max_rows=1)
error_Qzz_w = np.loadtxt( Input_dir + f'shear_moment_data_with_Qzz_{Fit_tech2}.txt', skiprows=13, usecols=2, max_rows=1)


print('For estimator wMLE the Qxx component of the shear moment is:',Qxx_data_w,'+-',error_Qxx_w,'km/s')
print('For estimator wMLE the Qyy component of the shear moment is:',Qyy_data_w,'+-',error_Qyy_w,'km/s')
print('For estimator wMLE the Qzz component of the shear moment is:',Qzz_data_w,'+-',error_Qzz_w,'km/s')
print('For estimator wMLE the Qxy component of the shear moment is:',Qxy_data_w,'+-',error_Qxy_w,'km/s')
print('For estimator wMLE the Qxz component of the shear moment is:',Qxz_data_w,'+-',error_Qxz_w,'km/s')
print('For estimator wMLE the Qyz component of the shear moment is:',Qyz_data_w,'+-',error_Qyz_w,'km/s')

For estimator wMLE the Qxx component of the shear moment is: 0.1302 +- 0.1782 km/s
For estimator wMLE the Qyy component of the shear moment is: -0.8355 +- 0.2666 km/s
For estimator wMLE the Qzz component of the shear moment is: 0.7054 +- 0.3206 km/s
For estimator wMLE the Qxy component of the shear moment is: -0.0078 +- 0.1542 km/s
For estimator wMLE the Qxz component of the shear moment is: -0.7917 +- 0.2879 km/s
For estimator wMLE the Qyz component of the shear moment is: -0.4743 +- 0.2714 km/s


In [ ]:
# extraire la matrice C_pq des datas : pour etaMLE
#result_eta = shear_data_calculator('/renoir/fromenti/Documents/data_DESI/combinedpv/Y1/PV_clustering_data_v5_v13.fits', Fit_tech1)
# l'odre de Q_cov est : Qxx, Qxy, Qxz,Qyy,Qyz
#Q_MC_eta   = result_eta["Q_MC"]  # shear moment calculated for plot, errors bars and contours
#Q_cov_eta  = result_eta["Q_cov"] 

#print(Q_cov_eta) # on prend cette matrice

In [230]:
# extraire la matrice C_pq des datas : pour wMLE
#result_w = shear_data_calculator('/renoir/fromenti/Documents/data_DESI/combinedpv/Y1/PV_clustering_data_v5_v13.fits', Fit_tech2)

#Q_MC_w   = result_w["Q_MC"]  # shear moment calculated for plot, errors bars and contours
#Q_cov_w = result_w["Q_cov"] 

#print(Q_cov_w)

In [236]:
def chi2(B_measured,B_pq, C_pq):
    
    B_measured_transposée = B_measured.T
    Tot_pq = B_pq + C_pq
    Tot_pq_inv = np.linalg.inv(Tot_pq)
    # ici la matrice B_measured est la matrice des datas
    chi2 = B_measured @ Tot_pq_inv @ B_measured_transposée
    
    return chi2

C_pq_shear_etaMLE = np.array([[ 0.028257 ,-0.000628 ,-0.006035, -0.018957, -0.002586] ,  # je l'ai réecrite ici afin de ne pas refaire tourner le bulk_flow_calculator à chaque fois
 [ -0.000628 , 0.023508 , 0.008119 ,-0.00373  , 0.008842],
 [-0.006035 , 0.008119 , 0.078114,  0.000127, -0.005283],
 [ -0.018957 ,-0.00373 ,  0.000127 , 0.06693  , 0.015829],
 [ -0.002586 , 0.008842, -0.005283 , 0.015829 , 0.071627]])

Q_eta = np.array([Qxx_data_eta,Qyy_data_eta,Qxy_data_eta,Qxz_data_eta,Qyz_data_eta]) # Up sans Qzz car plus simple et en plus le calule
chi2_norm_eta = chi2(Q_eta,R_shear,C_pq_shear_etaMLE) / 5 # on normalise le chi2

print('The chi2 of the prediction of the shear moment normalize versus the data for etaMLE is:',chi2_norm_eta)

The chi2 of the prediction of the shear moment normalize versus the data for etaMLE is: 2.999960920576131


In [237]:
C_pq_shear_wMLE = np.array([[0.029639 , 0.00009 , -0.004688, -0.01726 , -0.005093] ,  # je l'ai réecrite ici afin de ne pas refaire tourner le bulk_flow_calculator à chaque fois
 [0.00009  , 0.024939  ,0.00788 , -0.004991,  0.010001 ],
 [ -0.004688 , 0.00788  , 0.080467 ,-0.005361 , 0.002664],
 [ -0.01726 , -0.004991 ,-0.005361 , 0.066692,  0.00942 ],
 [ -0.005093 , 0.010001 , 0.002664  ,0.00942 ,  0.075467]])

Q_w = np.array([Qxx_data_w ,Qyy_data_w ,Qxy_data_w ,Qxz_data_w ,Qyz_data_w ]) # la matrice B qui est comme Up mais ici on prend que le bulk flow par simplicité
chi2_norm_w = chi2(Q_w,R_shear,C_pq_shear_wMLE) / 5  # on normalise le chi2

print('The chi2 of the prediction of the shear moment normalize versus the data for wMLE is:',chi2_norm_w)

The chi2 of the prediction of the shear moment normalize versus the data for wMLE is: 2.9820830294161853
